# Group G
dataset: https://www.kaggle.com/datasets/sobhanmoosavi/us-weather-events/data

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# This might be needed in order to get plots to display in the exported HTML for submission.
# In any case, please double check that plots display properly before you submit.
import plotly.io as pio
pio.renderers.default='notebook'
df = pd.read_csv("WeatherEvents_Jan2016-Dec2022.csv")

## Distribuzione Oraria degli Eventi Meteo

In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'], utc=True)
df['EndTime(UTC)'] = pd.to_datetime(df['EndTime(UTC)'], utc=True)

def to_local(group):
    tz_name = group.name 
    group['Local_Time'] = group['StartTime(UTC)'].dt.tz_convert(tz_name).dt.tz_localize(None)
    return group

hourly_counts = df.groupby('TimeZone', group_keys=False).apply(to_local)

hourly_counts['Local_Hour'] = hourly_counts['Local_Time'].dt.hour
hourly_counts = hourly_counts.groupby(['Local_Hour', 'Type']).size().reset_index(name='Event_Count')

In [ ]:


# Creiamo il grafico
fig = px.line(
    hourly_counts, 
    x = 'Local_Hour', 
    y = 'Event_Count', 
    color = 'Type',
    title = 'Distribuzione Oraria degli Eventi Meteo (Ora Locale)',
    labels = {'Local_Hour': 'Ora del Giorno (0-23)', 'Event_Count': 'Numero di Eventi'},
    template = 'plotly_white'
)

fig.update_layout(xaxis=dict(tickmode='linear', tick0=0, dtick=1))

fig.show()

## Intensità piogga negli anni
(lo toglieri)

In [ ]:

# 1. Preparazione dati (identica a prima)
rain = df[df["Type"] == "Rain"].copy()
rain["DurationHrs"] = (rain["EndTime(UTC)"] - rain["StartTime(UTC)"]).dt.total_seconds() / 3600

rain = rain[
    (rain["DurationHrs"] > 0) & 
    (rain["Precipitation(mm)"] > 0) & 
    rain["LocationLat"].notna() & 
    rain["LocationLng"].notna()
]

rain["Intensity"] = rain["Precipitation(mm)"] / rain["DurationHrs"]

agg = rain.groupby(
    ["Year", "AirportCode", "City", "State", "LocationLat", "LocationLng"]
).agg(
    EventCount=("EventId", "count"),
    AvgIntensity=("Intensity", "mean"),
    AvgDurationHr=("DurationHrs", "mean"),
    TotalPrecipIn=("Precipitation(mm)", "sum")
).reset_index().sort_values("Year")

p99 = agg["AvgIntensity"].quantile(0.99)
agg["IntensityClipped"] = agg["AvgIntensity"].clip(upper=p99)

# 2. Creazione mappa con Plotly Express
fig = px.scatter_geo(
    agg,
    lat="LocationLat",
    lon="LocationLng",
    color="IntensityClipped",
    size="EventCount",
    animation_frame="Year",
    hover_name="AirportCode",
    hover_data={
        "City": True, 
        "State": True,
        "LocationLat": False, 
        "LocationLng": False,
        "IntensityClipped": False,
        "AvgIntensity": ":.4f",
        "AvgDurationHr": ":.2f",
        "TotalPrecipIn": ":.2f"
    },
    color_continuous_scale="Blues",
    range_color=[0, p99],
    size_max=8,
    scope="usa",
    title="Intensità pioggia negli USA per stazione meteo",
    labels={"IntensityClipped": "Intensità (mm/hr)"}
)

# 3. Aggiornamento estetico (colori scuri come nel tuo esempio)
fig.update_layout(
    geo=dict(
        projection_type="albers usa",
        showland=True, landcolor="#1a1a2e",
        showlakes=True, lakecolor="#16213e",
        showcoastlines=True, coastlinecolor="#444",
        showsubunits=True, subunitcolor="#333",
        bgcolor="#0f0f1a"
    ),
    paper_bgcolor="#0f0f1a",
    plot_bgcolor="#0f0f1a",
    font=dict(color="white")
)

fig.show()

## Intensità pioggia per anni per stato
(toglierei anche questo frse)

In [ ]:

rain = df[df["Type"] == "Rain"].copy()

rain["DurationHrs"] = (rain["EndTime(UTC)"] - rain["StartTime(UTC)"]).dt.total_seconds() / 3600

rain = rain[
    (rain["DurationHrs"] > 0) & 
    (rain["Precipitation(mm)"] > 0) & 
    rain["State"].notna()
]

rain["Intensity"] = rain["Precipitation(mm)"] / rain["DurationHrs"]

agg = rain.groupby(["Year", "State"]).agg(
    EventCount=("EventId", "count"),
    AvgIntensity=("Intensity", "mean"),
    AvgDurationHr=("DurationHrs", "mean"),
    TotalPrecipIn=("Precipitation(in)", "sum")
).reset_index().sort_values("Year")

p99 = agg["AvgIntensity"].quantile(0.99)
agg["IntensityClipped"] = agg["AvgIntensity"].clip(upper=p99)

fig = px.choropleth(
    agg,
    locations="State",
    locationmode="USA-states",
    color="IntensityClipped",
    animation_frame="Year",
    hover_name="State",
    hover_data={
        "State": False,
        "IntensityClipped": False,
        "AvgIntensity": ":.4f",
        "AvgDurationHr": ":.2f",
        "TotalPrecipIn": ":.2f",
        "EventCount": True
    },
    color_continuous_scale="Blues",
    range_color=[0, p99],
    scope="usa",
    title="Intensità media pioggia negli USA per stato",
    labels={"IntensityClipped": "Intensità (mm/hr)"}
)

fig.update_traces(
    marker_line_width=0.5, 
    marker_line_color="#ffffff"
)

fig.update_layout(
    geo=dict(
        bgcolor="#0f0f1a",
        lakecolor="#16213e",
        showlakes=True,
        showcoastlines=False,
        subunitcolor="#555"
    ),
    paper_bgcolor="#0f0f1a",
    plot_bgcolor="#0f0f1a",
    font=dict(color="white"),
    margin=dict(l=0, r=0, t=60, b=90),
    height=620
)

fig.show()